# Credit Card Fraud Detection
### CSE 2793 – Machine Learning Workshop 1
**Team:** Akshyat Pattnaik | Abinash Nayak | Subhransu Dhar  
**Institution:** ITER, Siksha 'O' Anusandhan (Deemed to be University), Bhubaneswar

## Phase 1 – Data Loading & Initial Setup

In [ ]:
# ── Standard libraries ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score,
                              RocCurveDisplay)

# ── Imbalanced-learn ──────────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

# ── XGBoost ───────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

print('All libraries loaded successfully.')

In [ ]:
# Load dataset
# Download from: https://www.kaggle.com/mlg-ulb/creditcardfraud
# Place creditcard.csv in the ../data/ directory
df = pd.read_csv('../data/creditcard.csv')

print(f'Dataset shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head()

In [ ]:
# Class distribution
fraud_count   = df['Class'].sum()
legit_count   = len(df) - fraud_count
fraud_pct     = fraud_count / len(df) * 100

print(f'Legitimate transactions : {legit_count:,}  ({100 - fraud_pct:.4f}%)')
print(f'Fraudulent transactions : {fraud_count:,}  ({fraud_pct:.4f}%)')

## Phase 2 – Exploratory Data Analysis (EDA)

In [ ]:
# ── 2.1 Class Distribution Plot ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
df['Class'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'], ax=ax)
ax.set_title('Class Distribution (0 = Legit, 1 = Fraud)')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_xticklabels(['Legitimate', 'Fraud'], rotation=0)
plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── 2.2 Transaction Amount Distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df[df['Class'] == 0], x='Amount', bins=50,
             color='steelblue', ax=axes[0])
axes[0].set_title('Amount Distribution – Legitimate')
axes[0].set_xlim(0, 1000)

sns.histplot(data=df[df['Class'] == 1], x='Amount', bins=50,
             color='tomato', ax=axes[1])
axes[1].set_title('Amount Distribution – Fraud')
axes[1].set_xlim(0, 1000)

plt.tight_layout()
plt.savefig('../results/figures/amount_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── 2.3 Correlation Matrix ───────────────────────────────────────────────────
corr = df.corr()
plt.figure(figsize=(20, 16))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.3, annot=False)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../results/figures/correlation_matrix.png', dpi=150)
plt.show()

# Top features correlated with Class
print('Top 10 features correlated with Class:')
print(corr['Class'].drop('Class').abs().sort_values(ascending=False).head(10))

In [ ]:
# ── 2.4 Outlier Detection (Boxplots) ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df[df['Class'] == 0]['Amount'].plot(kind='box', ax=axes[0],
                                     color='steelblue',
                                     title='Amount – Legitimate')
df[df['Class'] == 1]['Amount'].plot(kind='box', ax=axes[1],
                                     color='tomato',
                                     title='Amount – Fraud')
plt.tight_layout()
plt.savefig('../results/figures/outlier_boxplot.png', dpi=150)
plt.show()

## Phase 3 – Preprocessing & Feature Engineering

In [ ]:
# ── 3.1 Feature Scaling ───────────────────────────────────────────────────────
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])

# Drop Time (sequence marker, not a fraud predictor)
df.drop(columns=['Time'], inplace=True)

print('Scaling applied. Time column dropped.')
print(f'Remaining features: {df.shape[1] - 1}  |  Target: Class')

In [ ]:
# ── 3.2 Train / Test Split (stratified) ──────────────────────────────────────
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train size : {X_train.shape[0]:,}  |  Fraud in train: {y_train.sum()}')
print(f'Test size  : {X_test.shape[0]:,}   |  Fraud in test : {y_test.sum()}')

In [ ]:
# ── 3.3 SMOTE – applied ONLY to training data ─────────────────────────────────
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE – Legit: {(y_train == 0).sum():,}  Fraud: {(y_train == 1).sum():,}')
print(f'After  SMOTE – Legit: {(y_train_res == 0).sum():,}  Fraud: {(y_train_res == 1).sum():,}')

## Phase 4 – Model Training

In [ ]:
# ── 4.1 Logistic Regression (Baseline) ───────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_res, y_train_res)
y_pred_lr = lr.predict(X_test)
print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['Legit', 'Fraud']))

In [ ]:
# ── 4.2 Random Forest ────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_res, y_train_res)
y_pred_rf = rf.predict(X_test)
print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Legit', 'Fraud']))

In [ ]:
# ── 4.3 XGBoost ──────────────────────────────────────────────────────────────
xgb = XGBClassifier(n_estimators=100, use_label_encoder=False,
                    eval_metric='logloss', random_state=42)
xgb.fit(X_train_res, y_train_res)
y_pred_xgb = xgb.predict(X_test)
print('=== XGBoost ===')
print(classification_report(y_test, y_pred_xgb, target_names=['Legit', 'Fraud']))

## Phase 5 – Results & Evaluation

In [ ]:
# ── 5.1 Confusion Matrices ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models      = [('Logistic Regression', y_pred_lr),
               ('Random Forest',       y_pred_rf),
               ('XGBoost',             y_pred_xgb)]

for ax, (name, preds) in zip(axes, models):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Legit', 'Fraud'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.savefig('../results/figures/confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# ── 5.2 ROC Curves ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for name, model in [('Logistic Regression', lr),
                    ('Random Forest',       rf),
                    ('XGBoost',             xgb)]:
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name)

ax.set_title('ROC Curves – All Models')
plt.tight_layout()
plt.savefig('../results/figures/roc_curves.png', dpi=150)
plt.show()

In [ ]:
# ── 5.3 Summary Table ────────────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

summary = []
for name, preds in models:
    summary.append({
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds), 4),
        'Recall'   : round(recall_score(y_test, preds), 4),
        'F1-Score' : round(f1_score(y_test, preds), 4),
        'AUC-ROC'  : round(roc_auc_score(y_test, preds), 4),
    })

results_df = pd.DataFrame(summary)
results_df.to_csv('../results/model_comparison.csv', index=False)
print(results_df.to_string(index=False))

In [ ]:
# ── 5.4 Feature Importance (Random Forest) ───────────────────────────────────
importances = pd.Series(rf.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top_features.plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.title('Top 15 Feature Importances – Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../results/figures/feature_importance.png', dpi=150)
plt.show()